# 62 — Career Advisor
**Goal:** Generate role-aware career guidance from resume + JD analysis.

Earlier chapters asked *"how well does this resume fit this job?"*; this chapter asks *"what should this person do next?"* The advisor combines the data-driven layers of the stack — resume parsing, skill-gap analysis against a target role (Ch. 53) — with LLM-generated natural-language recommendations, so the advice is grounded in the candidate's actual skills rather than generic career platitudes.

**Why it matters for resumes / ATS:** a score tells a candidate they are 50% matched; a recommendation tells them what to learn, what to certify, and what to build. That coaching layer is what turns an ATS from a gatekeeper into a service — and it only works if every suggestion traces back to a real gap in the data.

## 1. Advisor Architecture

The pipeline is a straight line with one branch: **parse** the resume into a structured schema, **analyze** skill gaps against the target role (reusing the Ch. 53 machinery), **generate** recommendations, and only then hand the results to the LLM for natural-language phrasing — every LLM claim is grounded in computed facts.

**What the code does:** prints the five-stage pipeline:
1. Parse resume — structured `ResumeSchema`
2. Analyze skill gaps vs target role (Notebook 53)
3. Generate recommendations — skill gaps to fill, certifications, project ideas, career trajectories
4. Use the LLM for natural language
5. Ground everything in real data (skills, experience)

The ordering is the point: recommendations come from the *analysis*, and the LLM is the last stage, not the first — put the model first and the advice drifts from the resume.

In [ ]:
print('''Career advisor pipeline:
1. Parse resume -> structured ResumeSchema
2. Analyze skill gaps vs target role (Notebook 53)
3. Generate recommendations:
   - Skill gaps to fill
   - Certifications to pursue
   - Project ideas to build
   - Career trajectory options
4. Use LLM for natural language recommendations
5. Ground everything in real data (skills, experience)''')

## 2. Recommendation Builder

`CareerAdvisor` is the deterministic core: a small knowledge base of career paths, each with `core` and `nice` skill lists, and an `analyze()` method that diffs the candidate's skills against the target path — no LLM required for the gap computation itself.

**What the code does:**
- `career_paths` — three roles (`data_scientist`, `ml_engineer`, `nlp_engineer`), each with core and nice-to-have skills; an unknown role returns `{}`, which yields empty gaps instead of a crash.
- `analyze(resume_skills, target_role)` — computes `core_gaps` (missing essentials) and `nice_gaps` (missing optional), then maps them to recommendations: `"Learn {gap}"` for core gaps (blockers), `"Consider {gap}"` for nice gaps (optional) — the verb encodes priority.

**Expected (verified by running):** with `["Python", "NLP"]` targeting `ml_engineer`, the output is `Core gaps: ['MLOps', 'Docker']`, `Nice-to-have: ['Kubernetes', 'CI/CD']`, and four recommendations — `Learn MLOps`, `Learn Docker`, `Consider Kubernetes`, `Consider CI/CD`. The split is deliberate: learn the blockers, consider the differentiators.

In [ ]:
class CareerAdvisor:
    def __init__(self):
        self.career_paths = {
            "data_scientist": {"core": ["Python", "SQL", "ML"], "nice": ["Deep Learning", "MLOps"]},
            "ml_engineer": {"core": ["Python", "MLOps", "Docker"], "nice": ["Kubernetes", "CI/CD"]},
            "nlp_engineer": {"core": ["Python", "NLP", "Transformers"], "nice": ["PyTorch", "BERT"]},
        }
    
    def analyze(self, resume_skills, target_role):
        path = self.career_paths.get(target_role, {})
        core_gaps = [s for s in path.get("core", []) if s not in resume_skills]
        nice_gaps = [s for s in path.get("nice", []) if s not in resume_skills]
        
        recs = []
        for gap in core_gaps:
            recs.append(f"Learn {gap} — essential for {target_role}")
        for gap in nice_gaps:
            recs.append(f"Consider {gap} — recommended for {target_role}")
        
        return {"gaps": {"core": core_gaps, "nice": nice_gaps}, "recommendations": recs}

advisor = CareerAdvisor()
result = advisor.analyze(["Python", "NLP"], "ml_engineer")
print(f"Core gaps: {result['gaps']['core']}")
print(f"Nice-to-have: {result['gaps']['nice']}")
for r in result['recommendations']:
    print(f"  - {r}")

## 3. LLM-Enhanced Recommendations

The deterministic gaps from section 2 are *data*; the LLM turns them into *advice*. The prompt contract is a three-part structure: a **system role** (career coach for data professionals), a **context block** with the computed facts (`role`, `skills`, `gaps`, `target_role`), and a **structured output spec** — three quick wins, two strategic moves, one moonshot.

**What the code does:** prints the prompt template, which follows Ch. 56's five-component pattern: system, context, and output format — and deliberately *no* few-shot examples, because career advice is open-ended and the numbered output spec (`1. ... 2. ... 3. ...`) is the only structure needed.

**Expected (with a key):** the call is expected to return exactly three numbered quick wins (skills learnable in under a month, derived from the gaps), two strategic moves (certifications, projects), and one long-term moonshot — the same `gaps` dict that produced `Learn MLOps` now appears as fluent prose, grounded in the same data.

In [ ]:
print('''LLM prompt for career recommendations:

System: You are a career coach for data professionals. 
Given a resume analysis, provide actionable career advice.

Context:
  Current role: {role}
  Skills: {skills}
  Gaps: {gaps}
  Target: {target_role}

Generate:
1. Three quick wins (skills to learn in < 1 month)
2. Two strategic moves (certifications, projects)
3. One moonshot (long-term career growth)''')

## Summary: Career advisor combines data-driven gap analysis with LLM-generated natural language recommendations.

**Compute the gaps deterministically; let the LLM write the advice — never the reverse.**

`CareerAdvisor.analyze()` produces the facts (core vs nice gaps, Learn vs Consider recommendations) from a small career-path knowledge base, and the LLM only rephrases those facts into structured coaching output (quick wins, strategic moves, moonshot). Because the model never computes the gaps, the advice cannot hallucinate them — it can only embellish what the data already said.

This chapter feeds Ch. 63, where the block's model choices are stress-tested: the same resume tasks are compared across models on quality, cost, and latency.